# House Price Analysis — Exploratory Data Analysis
### Project 02 | Data Analysis Roadmap | May 2026

---

## Project Goal
Analyze the Kaggle House Prices dataset to answer:
**What factors most strongly predict house sale prices?**

- **Dataset:** Kaggle House Prices: Advanced Regression Techniques — `train.csv`
- **Size:** 1,460 houses · 81 columns
- **Target variable:** `SalePrice` (house sale price in USD)

---

## Analysis Questions

| # | Question | Key Tool | Stat Test |
|---|---|---|---|
| Q1 | What does the SalePrice distribution look like? | Histogram + Log transform | Descriptive |
| Q2 | Does living area predict price? | Scatter plot | Pearson r |
| Q3 | How does quality rating affect price? | Box plot | Spearman r |
| Q4 | Which neighborhoods have the highest prices? | Horizontal bar chart | Descriptive |
| Q5 | Has price changed over the years? | Line/scatter chart | Pearson r |
| Q6 | Does TotalSF predict better than GrLivArea alone? | Scatter + Feature engineering | Pearson r comparison |
| Q7 | Which features correlate most with price? | Correlation heatmap | Pearson r matrix |

---

## Key Columns Used

| Column | Meaning |
|---|---|
| `SalePrice` | Target — house sale price in USD |
| `GrLivArea` | Above ground living area (sq ft) |
| `TotalBsmtSF` | Basement area (sq ft) |
| `OverallQual` | Overall quality rating (1–10) |
| `YearBuilt` | Year house was built |
| `Neighborhood` | Location within Ames, Iowa |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_csv('../data/raw/train.csv')
df.head()

In [ ]:
df.describe()

## Q1 — What does the SalePrice distribution look like?

**Goal:** Plot the distribution of house prices, check if it's skewed, and apply log transformation to make it more symmetric for ML.

In [ ]:
saleprice = df['SalePrice'].dropna()
df['log_saleprice'] = np.log1p(df['SalePrice'])

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(saleprice, bins=30, color='lightgreen', label='Sale Price')
plt.title("Sale Price Distribution")
plt.xlabel('Sale Price')
plt.ylabel('Number of Houses')
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/saleprice_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean: {df['SalePrice'].mean():.2f}")
print(f"Median: {df['SalePrice'].median():.2f}")

if df['SalePrice'].mean() > df['SalePrice'].median():
    print("Here Mean > Median: The distribution is right-skewed.")
elif df['SalePrice'].mean() < df['SalePrice'].median():
    print("Here Mean < Median: The distribution is left-skewed.")
else:
    print("Here Mean = Median: The distribution is symmetric.")


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df['log_saleprice'], bins=30, color='lightblue', label='Log(Sale Price)')
plt.title("Log(Sale Price) Distribution")
plt.xlabel('Log(Sale Price)')
plt.ylabel('Number of Houses')
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/saleprice_distribution_log.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean of LogSalePrice: {df['log_saleprice'].mean():.3f}")
print(f"Median of LogSalePrice: {df['log_saleprice'].median():.3f}")
print(f"Skewness after log: {df['log_saleprice'].skew():.3f}")

### Q1 — Conclusion

**What I found:**
- House prices are right-skewed. Mean was $180,921 and median was $163,000, so mean > median — that confirms right skew.
- Most houses are between $100k and $200k. A few expensive houses create that long tail on the right.

**What I did about it:**
- Applied log transformation using `np.log1p()` to make the distribution more symmetric.
- Skewness went from >1 (highly skewed) down to 0.121 — basically normal now.

**Why this matters:**
- ML models work better when the target variable is roughly normally distributed.
- Log transform is standard for price data. I'll use `log_saleprice` later for regression.

**Code I learned:**
| What I wanted to do | How I wrote it |
|---------------------|----------------|
| Log transform | `df['log_saleprice'] = np.log1p(df['SalePrice'])` |
| Check skewness | `df['log_saleprice'].skew()` |
| Plot histogram | `plt.hist(df['log_saleprice'], bins=30)` |

## Q2 — Does living area predict price?

**Goal:** Use a scatter plot to visualize the relationship between above-ground living area (GrLivArea) and SalePrice, then measure the strength of the correlation using Pearson's r.

In [ ]:
living_price_df = df[['GrLivArea', 'SalePrice']].dropna()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=living_price_df, x='GrLivArea', y='SalePrice', color='orange', alpha=0.7)
plt.title("GrLivArea vs SalePrice")
plt.xlabel('Above Ground Living Area (sq ft)')
plt.ylabel('Sale Price')
plt.tight_layout()
plt.savefig("../visuals/grlivarea_vs_saleprice.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r_value, p_value = stats.pearsonr(living_price_df['GrLivArea'], living_price_df['SalePrice'])
if p_value < 0.05:
    print(f"Significant correlation: r = {r_value:.3f}, p = {p_value:.3e}")
elif p_value >= 0.05:
    print(f"No significant correlation: r = {r_value:.3f}, p = {p_value:.3e}")

### Q2 — Conclusion

**What I was trying to figure out:**
Whether a house's above-ground living area (GrLivArea) can tell me anything about its sale price. Two continuous variables — so I needed a way to see both the pattern and the strength.

**Why I chose what I chose:**
- **Scatter plot, not heatmap:** Heatmap would only give me one number (r). A scatter plot lets me see the actual shape of the relationship — is it linear? Are there outliers? That matters more than just a correlation value.
- **Pearson, not Spearman:** Both variables are continuous and I wanted to measure linear relationship. Spearman is for ranks/ordinal data — not needed here.

**What I found:**
- Scatter plot shows a clear positive trend. Bigger living area = higher price. But I also spotted outliers — some big houses selling cheap, some small houses selling expensive.
- Pearson r = 0.709 (strong positive correlation)
- p-value = 4.52e-223 (basically zero — this isn't random chance)

**What this means:**
Living area alone explains a lot about price, but it's not the whole story. Those outliers tell me other factors matter too (maybe quality, location, basement, etc.).

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Scatter plot | `sns.scatterplot(data=df, x='GrLivArea', y='SalePrice', alpha=0.4)` |
| Pearson + p-value | `stats.pearsonr(x, y)` |
| Significance check | `if p_value < 0.05:` |

## Q3 — How does quality rating affect price?

**Goal:** Use a box plot to see how SalePrice changes across different OverallQual ratings (1-10), and measure correlation since quality is ordinal.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='OverallQual', y='log_saleprice', color='lightcoral')
plt.yscale('log')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.title("Overall Quality vs SalePrice")
plt.xlabel('Overall Quality')
plt.ylabel('Sale Price(log scale)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../visuals/overallqual_vs_saleprice.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
r_value_2, p_value_2 = stats.spearmanr(df['OverallQual'], df['log_saleprice'])
if p_value_2 < 0.05:
    print(f"Significant correlation: r = {r_value_2:.3f}, p = {p_value_2:.3e}")
elif p_value_2 >= 0.05:
    print(f"No significant correlation: r = {r_value_2:.3f}, p = {p_value_2:.3e}")

In [ ]:
print(df['OverallQual'].value_counts().sort_index())

### Q3 — Conclusion

**What I was trying to figure out:**
Whether a house's overall quality rating (1-10, ordinal) predicts sale price. Different from Q2 — here the x variable is ordered categories, not continuous.

**Why I chose what I chose:**
- **Box plot, not scatter:** Scatter plots need two continuous variables. OverallQual has only 10 discrete values — box plot shows distribution at each quality level.
- **Spearman, not Pearson:** The project guide said Pearson, but that's wrong. Pearson assumes continuous data. OverallQual is ordinal (ranks). Spearman works on ranks — perfect for this.
- **Log scale on y-axis:** Price ranges from $50k to $600k+. Raw scale would make low-quality boxes tiny and hard to compare. Log scale shows percentage differences, which matters more for price. Also matches my Q1 log transformation.

**What I found:**
- Spearman r = 0.810 (stronger than living area's 0.709 — my guess was right)
- p = 0.000 (significant)
- Box plot shows median price generally increases with quality — BUT quality 2 looks weird (median lower than quality 1)

**Why quality = 2 looks wrong:**
- Only 3 houses in quality 2 (checked with `.value_counts()`)
- Too few data points → unreliable box plot
- This taught me: always check sample sizes before trusting group comparisons

**What this means:**
Quality is the strongest predictor I've seen so far (r = 0.81). But low-quality ratings (1-3) have very few samples — conclusions about them aren't reliable.

**Code I learned:**
| What I wanted | How I wrote it |
|---------------|----------------|
| Box plot | `sns.boxplot(data=df, x='OverallQual', y='SalePrice')` |
| Log scale on y | `plt.yscale('log')` |
| Spearman correlation | `stats.spearmanr(x, y)` |
| Check sample sizes | `df['OverallQual'].value_counts().sort_index()` |

## Q4 — Does the neighborhood affect price?

**Goal:** Find median SalePrice for each neighborhood, sort from highest to lowest, and visualize with a horizontal bar chart.

In [26]:
df['Neighborhood']

0       CollgCr
1       Veenker
2       CollgCr
3       Crawfor
4       NoRidge
         ...   
1455    Gilbert
1456     NWAmes
1457    Crawfor
1458      NAmes
1459    Edwards
Name: Neighborhood, Length: 1460, dtype: str